# Chapter 02 · LangChain 첫걸음 — 실습 노트북

| 섹션 | 주제 |  
|------|------| 
| 01 | 환경 설정 — 노트북·conda·API 키 |  
| 02 | Building Blocks — 모델·메시지·프롬프트 |  
| 03 | LCEL — pipe·Runnable·체인 |  
| 04 | 로컬 모델 — Ollama·HF·최적화 |  
| 05 | 멀티모달 — 이미지 생성과 이해 |  

**사용법** — 위에서 아래로 셀을 순서대로 실행하세요. API 키가 필요한 셀은 키가 없으면 자동으로 건너뛰도록 가드를 넣어 두었습니다. 모든 셀을 다 실행할 필요는 없으며, 보유한 API 키/환경에 맞는 셀만 골라 실행해도 됩니다.

## 0. 사전 준비 — 패키지 설치

필요한 LangChain 패키지를 설치합니다. (이미 설치돼 있으면 건너뛰어도 됩니다.)

In [ ]:
# 핵심 패키지 설치 (필요한 것만 골라서 설치하세요)
%pip install -q \
    langchain langchain-core langchain-community \
    langchain-openai langchain-anthropic langchain-google-genai \
    langchain-ollama langchain-huggingface

### API 키 설정 (슬라이드 7–8)

슬라이드에서 소개한 **3가지 방법** 중 가장 안전한 방식(권장: 환경 변수 + `.gitignore`)에 가깝게,
여기서는 `os.environ` 으로 세션 동안만 키를 설정하는 헬퍼를 둡니다.

| Provider | 환경 변수 | 발급 |
|----------|-----------|------|
| OpenAI | `OPENAI_API_KEY` | platform.openai.com |
| Anthropic | `ANTHROPIC_API_KEY` | console.anthropic.com |
| Google | `GOOGLE_API_KEY` | aistudio.google.com |
| HuggingFace | `HUGGINGFACEHUB_API_TOKEN` | huggingface.co/settings |

> 책 예제는 `config.py` 의 `set_environment()` 를 사용합니다. 이 노트북은 단독 실행을 위해
> 아래처럼 직접 입력받는 방식으로 대체합니다. 키를 코드에 하드코딩하지 말고 입력 프롬프트를 쓰세요.

In [1]:
# setting the environment variables, the keys
from dotenv import load_dotenv
import sys
import os

load_dotenv()

True

---
## SECTION 01 · 환경 설정 

### LangChain이 지원하는 LLM 생태계 

LangChain의 핵심 가치는 **공급자를 바꿔 끼워도 같은 인터페이스**라는 점입니다.
설치된 패키지/보유 키에 따라 사용할 채팅 모델을 하나 고르는 헬퍼를 만들어 둡니다 —
이후 모든 섹션에서 재사용합니다.

In [2]:
def get_chat_model(temperature: float = 0.7):
    "보유한 키에 맞춰 ChatModel을 하나 반환. 우선순위: OpenAI > Anthropic > Google > Fake."
    if os.environ.get("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model="gpt-4o-mini", temperature=temperature)
    if os.environ.get("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model="claude-3-5-sonnet-latest", temperature=temperature)
    if os.environ.get("GOOGLE_API_KEY"):
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model="gemini-1.5-pro", temperature=temperature)
    # 키가 하나도 없으면 오프라인 테스트용 가짜 모델
    from langchain_community.llms import FakeListLLM
    print("⚠ API 키 없음 — FakeListLLM(데모용)로 대체합니다.")
    return FakeListLLM(responses=["(데모 응답) 키를 설정하면 실제 모델이 동작합니다."])

llm = get_chat_model()
type(llm).__name__

'ChatOpenAI'

---
## SECTION 02 · Building Blocks  

### 2-1. Model 인터페이스: LLM vs Chat  

`string → string` 의 **Legacy LLM** 인터페이스. LangChain 권장은 Chat 인터페이스지만,
통일된 `invoke()` 를 먼저 체험합니다. (키 없이도 `FakeListLLM` 으로 동작)

In [5]:
from langchain_core.language_models.fake import FakeListLLM

# 어떤 입력이든 동일 응답을 주는 가짜 LLM — 인터페이스 감 잡기용
fake_llm = FakeListLLM(responses=["Hello"])
print(fake_llm.invoke("Any input will return Hello"))

Hello


### 2-2. Chat 메시지 타입  

대화는 **역할(Role) + 내용(Content)** 으로 구조화합니다.
`SystemMessage`(행동 설정) · `HumanMessage`(사용자 입력) · `AIMessage`(모델 응답).

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You're a helpful programming assistant"),
    HumanMessage(content="Write a Python function to calculate factorial"),
]

llm = get_chat_model(temperature=0)
response = llm.invoke(messages)
print(getattr(response, "content", response))

Certainly! Here’s a simple Python function to calculate the factorial of a non-negative integer using recursion:

```python
def factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    elif n == 0 or n == 1:
        return 1
    else:
        return n * factorial(n - 1)

# Example usage:
print(factorial(5))  # Output: 120
```

Alternatively, you can also calculate the factorial using an iterative approach:

```python
def factorial(n):
    if n < 0:
        raise ValueError("Factorial is not defined for negative numbers.")
    
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result

# Example usage:
print(factorial(5))  # Output: 120
```

Both functions will correctly compute the factorial of a non-negative integer. You can choose either the recursive or iterative version based on your preference.


### 2-3. Reasoning 모델 — Extended Thinking  

Anthropic Claude의 **확장 사고(extended thinking)** 는 추론 토큰 예산을 명시적으로 제어합니다.
(Anthropic 키가 있을 때만 실행)

In [ ]:
if os.environ.get("ANTHROPIC_API_KEY"):
    from langchain_anthropic import ChatAnthropic

    thinking_model = ChatAnthropic(
        model="claude-3-7-sonnet-latest",
        max_tokens=4000,
        thinking={"type": "enabled", "budget_tokens": 2000},
    )
    resp = thinking_model.invoke("9.11과 9.9 중 어느 것이 더 큰가? 단계적으로 추론하라.")
    print(resp.content)
else:
    print("⏭  ANTHROPIC_API_KEY 없음 — 이 셀은 건너뜁니다.")

### 2-4. 모델 동작 제어 파라미터 

| Parameter | 설명 | 범위 | 쓰임 |
|-----------|------|------|------|
| Temperature | 출력의 무작위성 | 0.0–1.0+ | 낮음=사실 Q&A · 높음=창작 |
| Top-p | 누적 확률 threshold | 0.0–1.0 | 다양성 제어 |
| max_tokens | 최대 출력 길이 | 정수 | 비용·길이 제한 |

`temperature` 를 바꿔 가며 출력이 어떻게 달라지는지 직접 비교해 보세요.

In [8]:
prompt_text = "Give me one creative name for a coffee shop."

for t in (0.0, 1.0):
    m = get_chat_model(temperature=t)
    out = m.invoke(prompt_text)
    print(f"[temperature={t}] ->", getattr(out, "content", out))

[temperature=0.0] -> "Bean There, Brewed That"
[temperature=1.0] -> "The Brewed Awakening"


### 2-5. 프롬프트 템플릿 — f-string → PromptTemplate  

정적 문자열은 확장이 어렵습니다. `PromptTemplate` 은 변수를 분리해 재사용·테스트를 쉽게 합니다.

In [9]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template(
    "Summarize this text in one sentence:\n\n{text}"
)

# 변수만 채워서 프롬프트 완성
formatted = template.format(text="LangChain은 LLM 애플리케이션을 조립하는 프레임워크다.")
print(formatted)

Summarize this text in one sentence:

LangChain은 LLM 애플리케이션을 조립하는 프레임워크다.


### 2-6. Chat Prompt Templates  

역할별 메시지를 변수와 함께 구조화합니다 — 예: 영→독 번역기.

In [11]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ("system", "You are an English to German translator."),
    ("user", "Translate this to German: {text}"),
])

chat = get_chat_model(temperature=0)
formatted_messages = template.format_messages(text="Hello, how are you?")
result = chat.invoke(formatted_messages)
print(getattr(result, "content", result))

Hallo, wie geht es dir?


---
## SECTION 03 · LCEL — LangChain Expression Language  

### 3-1. pipe 연산자  

`prompt | llm | parser` — Unix 파이프처럼 컴포넌트를 선언적으로 조립합니다.

In [13]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 3-컴포넌트 체인: prompt -> llm -> parser
prompt = PromptTemplate.from_template("다음  {topic}에 대한 농담을 해줘")
llm = get_chat_model()
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

print(chain.invoke({"topic": "LLM"}))

물론이죠! 여기 LLM에 대한 농담 하나 있습니다:

왜 LLM은 항상 파티에 초대받을까요?

왜냐하면 언제나 좋은 "대화"를 만들어주니까요! 😄


### 3-2. Runnable 인터페이스 

모든 컴포넌트는 `invoke` · `stream` · `batch` · `ainvoke` 를 공유합니다.

### 3-3. 복잡한 체인: 컨텍스트 보존  

`RunnablePassthrough` 로 원본 입력(topic)을 잃지 않고 story·analysis를 함께 전달합니다.

In [15]:
from langchain_core.runnables import RunnablePassthrough

llm = get_chat_model()
parser = StrOutputParser()

story_prompt = PromptTemplate.from_template("다음 {topic}에 대한 이야기를 만들어줘.")
analysis_prompt = PromptTemplate.from_template(
    "Analyze the following story's mood:\n{story}"
)

story_chain = story_prompt | llm | parser
analysis_chain = analysis_prompt | llm | parser

# topic 은 그대로 흘리고, story 키를 새로 추가
combined = (
    RunnablePassthrough.assign(story=story_chain)
    | RunnablePassthrough.assign(analysis=analysis_chain)
)

out = combined.invoke({"topic": "a robot learning to paint"})
print("STORY:\n", out["story"][:300], "...\n")
print("ANALYSIS:\n", out["analysis"][:300], "...")

STORY:
 제목: 로봇의 첫 번째 그림

한때, 먼 미래의 도시에는 예술이 인간의 감정과 창의력을 표현하는 중요한 수단으로 여겨졌다. 하지만 이제는 로봇들이 인간의 예술가와 함께 작업하는 시대가 도래했다. 그 중에서도 특별한 로봇, 이름하여 '아르트로'가 있었다. 아르트로는 인공지능으로 설계된 로봇이었고, 인간의 감정을 이해하고 표현할 수 있도록 훈련받았다.

아르트로는 처음으로 그림을 그리는 날을 기다렸다. 그의 프로그램에는 다양한 예술 스타일과 기법, 그리고 색채 이론이 포함되어 있었지만, 실제로 캔버스에 그림을 그리는 것은 처음이었다.  ...

ANALYSIS:
 이 이야기의 전반적인 분위기는 긍정적이고 희망적인 느낌을 전달합니다. 여러 요소를 통해 감정이 잘 드러나며, 로봇 아르트로의 성장과 발견 과정을 통해 인간의 감정, 창의성, 그리고 예술의 본질에 대한 탐구가 이루어집니다.

1. **기대와 긴장감**: 이야기의 초반부에서 아르트로가 첫 번째 그림을 그리는 날을 기다리는 장면은 기대감과 긴장감을 드러냅니다. 로봇이 처음으로 예술을 창조하려는 노력은 독자에게 흥미로운 관심을 불러일으킵니다.

2. **자아 발견**: 아르트로가 자신의 감정을 표현해야 한다는 사실을 깨닫는 순간은 매우 중 ...


---
## SECTION 04 · 로컬 모델  

### 4-1. Ollama — 가장 친숙한 로컬 옵션  

먼저 터미널에서 모델을 받고 서버를 띄웁니다:

```bash
ollama pull deepseek-r1:1.5b
ollama serve
pip install langchain-ollama
```

그다음 LCEL 로 연결합니다. (로컬에 Ollama 서버가 떠 있을 때만 동작)

In [ ]:
try:
    from langchain_ollama import ChatOllama

    chat = ChatOllama(model="deepseek-r1:1.5b", temperature=0)
    messages = [
        ("system", "You are a helpful assistant."),
        ("human", "What makes LangChain great for working with LLMs?"),
    ]
    ai_msg = chat.invoke(messages)
    print(ai_msg.content)
except Exception as e:
    print("⏭  Ollama 미실행/미설치 — 건너뜁니다.\n   ", e)

### 4-2. Hugging Face 로컬 실행 (슬라이드 26)

`HuggingFacePipeline` 로 자체 인프라에서 직접 추론합니다. (작은 모델이라도 첫 다운로드는 느릴 수 있음)

In [19]:
!pip install transformers torch transformers accelerate

  Using cached torch-2.12.0-cp312-cp312-win_amd64.whl.metadata (31 kB)
  Using cached setuptools-81.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached torch-2.12.0-cp312-cp312-win_amd64.whl (123.0 MB)
Using cached setuptools-81.0.0-py3-none-any.whl (1.1 MB)
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)

   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [mpmath]
   ---------------------------------------- 0/6 [m

In [3]:
try:
    from langchain_core.messages import SystemMessage, HumanMessage
    from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline

    llm_hf = HuggingFacePipeline.from_model_id(
        model_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        task="text-generation",
        pipeline_kwargs=dict(max_new_tokens=256, do_sample=False, repetition_penalty=1.03),
    )
    chat_model = ChatHuggingFace(llm=llm_hf)
    resp = chat_model.invoke([
        SystemMessage(content="You're a helpful assistant"),
        HumanMessage(content="In one sentence, what is LangChain?"),
    ])
    print(resp.content)
except Exception as e:
    print("⏭  HuggingFace 로컬 실행 건너뜀 (transformers/torch 필요):\n   ", e)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

C:\Users\moonj\OneDrive\012.AI.Claude\Projects\AI Agent\generative_ai_with_langchain-second_edition\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\moonj\.cache\huggingface\hub\models--TinyLlama--TinyLlama-1.1B-Chat-v1.0. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


<|system|>
You're a helpful assistant</s>
<|user|>
In one sentence, what is LangChain?</s>
<|assistant|>
LangChain is a language learning platform that offers personalized language courses and resources for learners of English, French, Spanish, and German.


### 4-3. 로컬 모델 운영 팁 — 에러 핸들링 

양자화(4-bit)로 메모리를 절감하고, 재시도 로직으로 일시적 실패를 방어합니다.

In [4]:
import time

def safe_model_call(llm, prompt, retries: int = 3, delay: float = 1.0):
    "일시적 오류에 대비한 재시도 래퍼."
    for attempt in range(1, retries + 1):
        try:
            return llm.invoke(prompt)
        except Exception as e:
            print(f"  시도 {attempt}/{retries} 실패: {e}")
            if attempt == retries:
                raise
            time.sleep(delay)

# 데모: 정상 모델로 호출
demo = get_chat_model(temperature=0)
print(getattr(safe_model_call(demo, "Say OK"), "content", "OK"))

OK!


---
## SECTION 05 · 멀티모달 AI  

### 5-1. Text-to-Image — DALL·E 

`DallEAPIWrapper` 로 텍스트에서 이미지를 생성합니다. (OpenAI 키 필요)

In [ ]:
if os.environ.get("OPENAI_API_KEY"):
    from langchain_community.utilities.dalle_image_generator import DallEAPIWrapper

    dalle = DallEAPIWrapper(model="dall-e-3", size="1024x1024", quality="standard", n=1)
    image_url = dalle.run("A detailed technical diagram of an AI agent")
    print("이미지 URL:", image_url)

    # 노트북에 미리보기
    try:
        from IPython.display import Image, display
        display(Image(url=image_url))
    except Exception:
        pass
else:
    print("⏭  OPENAI_API_KEY 없음 — 이미지 생성 건너뜁니다.")

### 5-2. 이미지 이해 (Image Understanding) (슬라이드 31)

ChatModel 인터페이스에 **텍스트 + 이미지** 를 함께 넣어 멀티모달 추론을 수행합니다.

In [10]:
import base64
import httpx

def to_data_url(image_source: str) -> str:
    "URL 또는 로컬 경로의 이미지를 base64 data URL로 변환 (OpenAI가 직접 다운로드하지 않도록)."
    if image_source.startswith(("http://", "https://")):
        resp = httpx.get(image_source, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
        resp.raise_for_status()
        data = resp.content
        mime = resp.headers.get("content-type", "image/jpeg").split(";")[0]
    else:
        with open(image_source, "rb") as fh:
            data = fh.read()
        mime = "image/jpeg"
    b64 = base64.b64encode(data).decode()
    return f"data:{mime};base64,{b64}"

def analyze_image(image_source: str, question: str) -> str:
    if not os.environ.get("OPENAI_API_KEY"):
        return "⏭  OPENAI_API_KEY 없음 — 건너뜀"
    from langchain_core.messages import HumanMessage
    from langchain_openai import ChatOpenAI

    data_url = to_data_url(image_source)   # 함수 안에서 base64 변환
    chat = ChatOpenAI(model="gpt-4o-mini", max_tokens=256)
    message = HumanMessage(content=[
        {"type": "text", "text": question},
        {"type": "image_url", "image_url": {"url": data_url}},
    ])
    return chat.invoke([message]).content

# 호출은 이렇게 — URL 문자열과 질문 문자열만 넘기면 됩니다
url = "https://raw.githubusercontent.com/moonjukhim/ai-agent/main/02/image/4a2476a5-5064-4c06-8029-356e7ed4c0e1.jpg"
print(analyze_image(url, "이 이미지에 무엇이 보이나요? 한국어로 설명하세요."))

이미지에는 녹색 스포츠카가 보입니다. 차는 낮고 유선형의 디자인을 가지고 있으며, 후면에 여러 개의 슬릿이 있는 형태입니다. 바퀴는 둥글고, 바퀴와 차체의 조화가 잘 어우러져 있습니다. 자동차의 전반적인 분위기는 우아하고 고급스러운 느낌을 줍니다. 바디의 색상은 짙은 녹색으로, 매력적인 광택이 있습니다.


---
## CHAPTER SUMMARY 

1. **환경 셋업** — Colab/conda + API 키 관리(`config.py` + `.gitignore`). LangChain은 OpenAI·Anthropic·Google·Replicate 등을 공통 API로 묶는다.
2. **Building Blocks** — Chat 인터페이스 + System/Human/AI 메시지. Temperature·top-p·max_tokens 로 동작 제어.
3. **LCEL** — `prompt | llm | parser` 파이프로 선언적 조립. 모든 컴포넌트가 Runnable(invoke·stream·batch·ainvoke).
4. **로컬 모델** — Ollama·HuggingFace 로 프라이버시·비용 제어. 양자화·재시도로 운영.
5. **멀티모달** — 이해 vs 생성. DALL·E/Stable Diffusion 생성, ChatModel 로 이미지 이해.

### 복습 문제  

> 아래 마크다운 셀에 직접 답을 적어 보며 정리하세요.

- **Q1.** LCEL의 목적을 가장 잘 설명하는 것은?
- **Q2.** LLM과 Chat 모델의 인터페이스·사용 케이스 차이는?
- **Q3.** Runnable이 모듈화된 LLM 앱 구축에 하는 역할은?
- **Q4.** 로컬 모델 성능에 영향을 주는 요인은?
- **Q5.** Cloud · llama.cpp · GPT4All 중 상황별 선택 기준은?
- **Q6.** 멀티모달 '이해'와 '생성'의 차이는?